## Introduction

This notebook demonstrates the use of **scikit‑learn’s `ColumnTransformer`** to streamline the preprocessing of heterogeneous tabular data. The dataset (`covid_toy.csv`) contains **500 synthetic records** with the following columns:

- **age** (numeric)  
- **gender** (categorical: Male/Female)  
- **fever** (numeric, with ~16% missing values)  
- **cough** (ordinal: Mild/Strong)  
- **city** (categorical: Kolkata, Bangalore, Delhi, Mumbai)  
- **has_covid** (target: Yes/No)

The notebook contrasts two approaches:

1. **Without `ColumnTransformer`** – manually handling each column with separate transformers, then concatenating the results.  
2. **With `ColumnTransformer`** – defining a single transformer pipeline that applies the appropriate transformations to each column in one go.

The goal is to highlight how `ColumnTransformer` reduces code complexity, avoids index mismatches, and makes the preprocessing step more maintainable and less error‑prone, especially when dealing with mixed data types.

In [ ]:
# Importing Modules and Packages 

import pandas as pd
import numpy as np


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder



In [5]:
df = pd.read_csv("../data/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,47,Female,98.4,Strong,Delhi,No
1,37,Female,98.2,Mild,Bangalore,No
2,49,Male,NaN,Mild,Mumbai,No
3,62,Female,98.5,Mild,Bangalore,No
4,36,Female,102.7,Mild,Mumbai,No


In [10]:
df['city'].value_counts()

city
Bangalore    163
Kolkata      134
Delhi        126
Mumbai        77
Name: count, dtype: int64

In [11]:
df['cough'].value_counts()

cough
Mild      300
Strong    200
Name: count, dtype: int64

In [7]:
df.isnull().sum()

age           0
gender        0
fever        80
cough         0
city          0
has_covid     0
dtype: int64

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=45)
print(f"Shape of Training set X : {X_train.shape}")
print(f"Shape of Test set X : {X_test.shape}")

Shape of Training set X : (400, 5)
Shape of Test set X : (100, 5)


In [13]:
X_train

,age,gender,fever,cough,city
484,28,Female,98.7,Strong,Kolkata
166,52,Female,98.5,Strong,Bangalore
455,28,Female,NaN,Mild,Mumbai
63,22,Male,98.7,Mild,Bangalore
180,49,Female,NaN,Strong,Bangalore
...,...,...,...,...,...
32,39,Male,NaN,Strong,Bangalore
380,27,Male,103.8,Strong,Bangalore
131,41,Female,98.7,Strong,Mumbai
414,44,Male,98.6,Mild,Mumbai


## 1. Without using Column Transformer

#### 1.1 Handling Missing values in 'fever'

In [ ]:
Si = SimpleImputer()

## Fit on train, transform both!
Si.fit(X_train[['fever']])

X_train_fever = Si.transform(X_train[['fever']]) # return numpy array
X_test_fever = Si.transform(X_test[['fever']])


X_train_fever.shape


(400, 1)

#### 1.2 OrdinalEncoding for 'cough' column

In [ ]:
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
oe.fit(X_train[['cough']]) # fit on train, transform both

X_train_cough = oe.transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])


X_train_cough.shape


(400, 1)

#### 1.3 OneHotEncoding for 'gender' and 'city'


In [ ]:
ohe = OneHotEncoder(drop='first', sparse_output=False)

ohe.fit(X_train[['gender' , 'city']])

X_train_gen_city = ohe.transform(X_train[['gender', 'city']])
X_test_gen_city = ohe.transform(X_test[['gender', 'city']])

X_train_gen_city

array([[0., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 1.],
       ...,
       [0., 0., 0., 1.],
       [1., 0., 0., 1.],
       [0., 0., 1., 0.]])

#### 1.4 Extracting Age 


In [35]:
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values # Converting into numpy array

X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_test_age.shape


(100, 1)

In [43]:
X_train.index

Index([484, 166, 455,  63, 180, 368,  22, 479, 228, 285,
       ...
        68, 163, 213, 445,  95,  32, 380, 131, 414, 459],
      dtype='int64', length=400)

#### 1.5 Concatenation 

In [45]:
cols = ['age'] + ohe.get_feature_names_out(['gender', 'city']).tolist() + ['fever' , 'cough']

X_train_transformed = pd.DataFrame(
    
    np.concatenate((X_train_age, X_train_gen_city, X_train_fever, X_train_cough), axis=1),
    columns=cols,
    index=X_train.index
    
)

X_test_transformed = pd.DataFrame(
    
    np.concatenate((X_test_age, X_test_gen_city, X_test_fever, X_test_cough), axis=1),
    columns=cols,
    index=X_test.index
    
)

X_train_transformed

,age,gender_Male,city_Delhi,city_Kolkata,city_Mumbai,fever,cough
484,28.0,0.0,0.0,1.0,0.0,98.700000,1.0
166,52.0,0.0,0.0,0.0,0.0,98.500000,1.0
455,28.0,0.0,0.0,0.0,1.0,99.738623,0.0
63,22.0,1.0,0.0,0.0,0.0,98.700000,0.0
180,49.0,0.0,0.0,0.0,0.0,99.738623,1.0
...,...,...,...,...,...,...,...
32,39.0,1.0,0.0,0.0,0.0,99.738623,1.0
380,27.0,1.0,0.0,0.0,0.0,103.800000,1.0
131,41.0,0.0,0.0,0.0,1.0,98.700000,1.0
414,44.0,1.0,0.0,0.0,1.0,98.600000,0.0


## 2. Using Column Transformer


In [53]:
from sklearn.preprocessing import StandardScaler

In [63]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(transformers=[
    ('tnf1',StandardScaler(), ['age'] ),
    ('tnf2', SimpleImputer(), ['fever']),
    ('tnf3', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf4', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
    
    ], remainder='passthrough') # Remainder = 'passthrough' --> leave remaining columns as it is or 'drop' --> drop remaining columns


In [ ]:
# Fit on train, transform both!
transformer.fit(X_train)

# Transforming Both Train and Test data
X_train_tran = transformer.transform(X_train)
X_test_tran = transformer.transform(X_test)


# Converting into Pandas dataFrame
cols = ['age'] +['fever'] + ['cough'] + ohe.get_feature_names_out(['gender', 'city']).tolist()

X_train_transform = pd.DataFrame(
    
    X_train_tran,
    columns=cols,
    index=X_train.index

)

X_test_transform = pd.DataFrame(
    
    X_test_tran,
    columns=cols,
    index=X_test.index

)

X_test_transform


,age,fever,cough,gender_Male,city_Delhi,city_Kolkata,city_Mumbai
204,-1.484249,100.900000,1.0,0.0,0.0,1.0,0.0
481,-1.053616,98.900000,0.0,0.0,0.0,0.0,1.0
296,0.956006,99.738623,1.0,1.0,0.0,0.0,0.0
38,-1.412477,98.600000,1.0,1.0,0.0,1.0,0.0
298,0.884234,99.738623,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
79,-1.556022,98.300000,0.0,0.0,0.0,1.0,0.0
476,0.597145,102.800000,0.0,0.0,0.0,1.0,0.0
69,-0.694755,102.400000,0.0,0.0,0.0,1.0,0.0
73,1.673728,99.738623,1.0,0.0,1.0,0.0,0.0


## Key Notes & Observations

- **Manual approach (without `ColumnTransformer`):**  
  - Each column is processed individually:  
    - `age` extracted as a NumPy array.  
    - `fever` imputed with `SimpleImputer` (using the training set’s mean).  
    - `cough` ordinal‑encoded with `OrdinalEncoder` (Mild → 0, Strong → 1).  
    - `gender` and `city` one‑hot encoded with `OneHotEncoder` (dropping the first category to avoid multicollinearity).  
  - All transformed pieces are then concatenated using `np.concatenate()`, requiring careful alignment of columns and indices.  
  - **Drawbacks:** verbose, error‑prone (column order, index preservation), and difficult to maintain when the pipeline grows.

- **Using `ColumnTransformer`:**  
  - A single object encapsulates all transformations:  
    - `StandardScaler` for `age`.  
    - `SimpleImputer` for `fever`.  
    - `OrdinalEncoder` for `cough`.  
    - `OneHotEncoder` for `gender` and `city`.  
  - The `remainder='passthrough'` option keeps any unmentioned columns (none in this case) as‑is.  
  - The transformer is `fit()` on the training set and then `transform()` both train and test sets in one call.  
  - The result is a single NumPy array that can be easily converted back to a DataFrame with proper column names.

- **Benefits of `ColumnTransformer`:**  
  - **Cleaner code** – all preprocessing logic is defined in one place.  
  - **Consistency** – the same transformations are applied to train and test without extra bookkeeping.  
  - **Integration** – works seamlessly with scikit‑learn pipelines (e.g., `Pipeline` + `ColumnTransformer`).  
  - **Performance** – can take advantage of parallelism (if `n_jobs` is set) for large datasets.

- **Important consideration:**  
  - The notebook uses `SimpleImputer` without specifying a strategy – default is `'mean'` (which is appropriate for numeric `fever`).  
  - For `OrdinalEncoder`, the category order is explicitly given as `[['Mild', 'Strong']]` to enforce the desired mapping.  
  - `OneHotEncoder` with `drop='first'` reduces dimensionality.

## Final Verdict

So, it  clearly demonstrates that **`ColumnTransformer` is the recommended approach** for preprocessing mixed‑type datasets in scikit‑learn. Compared to the manual concatenation method, it:

- **Reduces boilerplate code** – all transformations are declared in a concise, readable dictionary‑like structure.  
- **Eliminates common pitfalls** – such as misaligned indices or wrong column ordering.  
- **Enhances reproducibility** – the entire preprocessing logic is encapsulated, making it easier to share, reuse, and deploy (e.g., inside a `Pipeline` for cross‑validation).  

While the manual approach works, it becomes cumbersome as the number of features grows. The `ColumnTransformer` version is not only more elegant but also more robust, especially when integrating with model training and hyperparameter tuning.  

**Recommendation:** Adopt `ColumnTransformer` in all preprocessing pipelines to future‑proof your code and improve maintainability.